In [7]:
# load data
import os
import json


data_path = "data_finetune/kalamang_rule"

os.listdir(data_path)

['kalamang_all_batch_prompts_ADJ_400_openai_batch_results_10_subset.json',
 'kalamang_all_batch_prompts_ADJ_400_openai_batch_results_20_subset.json',
 'kalamang_all_batch_prompts_ADJ_400_openai_batch_results_5_subset.json',
 'kalamang_all_batch_prompts_ADV_400_openai_batch_results_10_subset.json',
 'kalamang_all_batch_prompts_ADV_400_openai_batch_results_20_subset.json',
 'kalamang_all_batch_prompts_ADV_400_openai_batch_results_5_subset.json',
 'kalamang_all_batch_prompts_NOUN-PNOUN_400_openai_batch_results_10_subset.json',
 'kalamang_all_batch_prompts_NOUN-PNOUN_400_openai_batch_results_20_subset.json',
 'kalamang_all_batch_prompts_NOUN-PNOUN_400_openai_batch_results_5_subset.json',
 'kalamang_all_batch_prompts_VERB_400_openai_batch_results_10_subset.json',
 'kalamang_all_batch_prompts_VERB_400_openai_batch_results_20_subset.json',
 'kalamang_all_batch_prompts_VERB_400_openai_batch_results_5_subset.json',
 'train.tsv',
 'valid.tsv']

In [8]:
train_set = os.path.join(data_path, "kalamang_all_batch_prompts_NOUN-PNOUN_400_openai_batch_results_20_subset.json")

with open(train_set, "r") as f:
    train_data = json.load(f)

In [9]:
train_data[0]

{'custom_id': 'kalamang_all_batch_prompts_NOUN-PNOUN_400_1',
 'generated_sentence': 'bal se sor=at koraru bunga iam bunga=obj bite',
 'generated_eng_translation': 'The flower has bitten the flower.'}

In [10]:
# Load kalamang parallel sents to make test set
kalamang_parallel_sents = "parallel_sents/kalamang_parallel_sentences_with_pos_and_unimorph.json"

with open(kalamang_parallel_sents, "r") as f:
    kalamang_data = json.load(f)

In [11]:
# Create train.tsv using train_data
with open("data_finetune/kalamang_rule/train.tsv", "w", encoding="utf-8") as f:
    for item in train_data:
        sentence = item["generated_sentence"]
        labels = item["generated_eng_translation"]
        f.write(f"{sentence}\t{labels}\n")

In [12]:
kalamang_data[0]

{'page': 52,
 'IGT': '',
 'spacy_info': {'res': [{'word': '[',
    'lemma': '[',
    'upos': 'X',
    'tag': 'XX',
    'morph': {}},
   {'word': 'stim2_3:45',
    'lemma': 'stim2_3:45',
    'upos': 'NUM',
    'tag': 'CD',
    'morph': {'NumType': 'Card'}},
   {'word': ']',
    'lemma': ']',
    'upos': 'PUNCT',
    'tag': '-RRB-',
    'morph': {'PunctSide': 'Fin', 'PunctType': 'Brck'}},
   {'word': '‘',
    'lemma': "'",
    'upos': 'PUNCT',
    'tag': '``',
    'morph': {'PunctSide': 'Ini', 'PunctType': 'Quot'}},
   {'word': 'The',
    'lemma': 'the',
    'upos': 'DET',
    'tag': 'DT',
    'morph': {'Definite': 'Def', 'PronType': 'Art'}},
   {'word': 'dog',
    'lemma': 'dog',
    'upos': 'NOUN',
    'tag': 'NN',
    'morph': {'Number': 'Sing'}},
   {'word': 'has',
    'lemma': 'have',
    'upos': 'AUX',
    'tag': 'VBZ',
    'morph': {'Mood': 'Ind',
     'Number': 'Sing',
     'Person': '3',
     'Tense': 'Pres',
     'VerbForm': 'Fin'}},
   {'word': 'bitten',
    'lemma': 'bite',
 

In [16]:
with open("data_finetune/kalamang_rule/train.tsv", "r", encoding="utf-8") as f:
    for line in f:
        if '\t' not in line:
            print("Problematic line:", line)


Problematic line: kewe-an eir rombongan=plnk two"

Problematic line: 

Problematic line: Step 5: The English translation of the new sentence is: "I have two groups."

Problematic line: 

Problematic line: Final Output:



In [13]:
# Create valid.tsv using kalamang_data, taking the first 400 as train and the rest as valid
with open("data_finetune/kalamang_rule/valid.tsv", "w", encoding="utf-8") as f:
    for item in kalamang_data[400:]:
        sentence = item["sentence"]
        labels = item["translated_sentence"]
        f.write(f"{sentence}\t{labels}\n")

In [14]:
import pandas as pd
from datasets import Dataset
from transformers import (
    MarianTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    MarianMTModel,
)
from peft import LoraConfig, get_peft_model

# -------------------------
# Load Data
# -------------------------
def load_tsv(path):
    df = pd.read_csv(path, sep="\t", header=None, names=["src", "tgt"])
    return Dataset.from_pandas(df)

train_ds = load_tsv("data_finetune/kalamang_rule/train.tsv")
valid_ds = load_tsv("data_finetune/kalamang_rule/valid.tsv")

# -------------------------
# Load Model
# -------------------------
model_name = "Helsinki-NLP/opus-mt-en-ROMANCE"   # <-- replace with your closest pair
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# -------------------------
# LoRA Config (recommended for small data)
# -------------------------
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

# -------------------------
# Preprocessing
# -------------------------
max_length = 128

def preprocess(batch):
    inputs = tokenizer(
        batch["src"], max_length=max_length, truncation=True, padding="max_length"
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["tgt"], max_length=max_length, truncation=True, padding="max_length"
        )
    inputs["labels"] = labels["input_ids"]
    return inputs




c:\ProgramData\miniconda3\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


trainable params: 589,824 || all params: 78,533,120 || trainable%: 0.7511


In [15]:
print(train_ds[:10])


{'src': ['bal se sor=at koraru bunga iam bunga=obj bite', 'bal se sor=at koraru Tanamera iam Tanamera=obj bite', 'bal se sor=at koraru kulun iam fish=obj bite', 'kalamang sentence', 'bal se sor=at koraru sawarer iam fish=obj bite', 'bal se sor=at koraru am perun iam am perun=obj bite', 'bal se sor=at koraru tadon iam tadon=obj bite', 'bal se sor=at koraru kalis sasarawe iam kalis sasarawe=obj bite', 'bal se sor=at koraru ulan iam fish=obj bite', 'bal se sor=at koraru goni iam fish=obj bite'], 'tgt': ['The flower has bitten the flower.', "[stim2_3:45] 'Tanamera has bitten Tanamera.'", 'The skin has bitten the fish.', 'English translation', 'The tortoise has bitten the fish.', 'The breast milk has bitten the breast milk.', 'The cough has bitten the cough.', 'The drizzle has bitten the light rain.', 'The aunt has bitten the fish.', 'The sack has bitten the fish.']}


In [3]:

train_ds = train_ds.map(preprocess, batched=True)
valid_ds = valid_ds.map(preprocess, batched=True)


Map:   0%|          | 0/7504 [00:00<?, ? examples/s]

c:\ProgramData\miniconda3\Lib\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


ValueError: Input is not valid. Should be a string, a list/tuple of strings or a list/tuple of integers.

In [ ]:
# -------------------------
# Training Arguments
# -------------------------
training_args = Seq2SeqTrainingArguments(
    output_dir="./marian-finetuned",
    evaluation_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,               # small dataset → more epochs OK
    predict_with_generate=True,
    fp16=True,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# -------------------------
# Trainer
# -------------------------
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


In [ ]:
# -------------------------
# Train
# -------------------------
trainer.train()

# Save final model
trainer.save_model("./marian-final")
